# Sentiment Analysis model training and evaluating with stars as features

### Importing all dependencies

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
import pickle
import warnings
warnings.filterwarnings('ignore')
import spacy

### Custom text preprocessor for cleaning and preparing review text

In [ ]:
nlp = spacy.load("en_core_web_sm")

class SpacyTextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, remove_stopwords=True):
        self.remove_stopwords = remove_stopwords

    def simple_tokenize(self, text):
        doc = nlp(text)
        return [token.text.lower() for token in doc if token.is_alpha]

    def simple_lemmatize(self, word):
        return nlp(word)[0].lemma_

    def clean_text(self, text):
        if not text:
            return ""
        doc = nlp(text.lower())
        tokens = [
            token.lemma_ for token in doc
            if token.is_alpha and (not self.remove_stopwords or not token.is_stop)
        ]
        return " ".join(tokens)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.clean_text(text) for text in X]

In [ ]:
class FeatureCombiner(BaseEstimator, TransformerMixin):
    def __init__(self, star_weight=0.01):
        self.star_weight = star_weight
        self.text_vectorizer = TfidfVectorizer(
            max_features=50000,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        self.scaler = StandardScaler()
        self.fitted = False

    def fit(self, X, y=None):
        texts = [item[0] for item in X]
        stars = np.array([item[1] for item in X]).reshape(-1, 1)

        self.text_vectorizer.fit(texts) # Fit text vectorizer

        self.scaler.fit(stars) # Fit scaler for stars

        self.fitted = True
        return self

    def transform(self, X):
        if not self.fitted:
            raise ValueError("Must fit the transformer before transform")

        texts = [item[0] for item in X]
        stars = np.array([item[1] for item in X]).reshape(-1, 1)

        # Transform text to TF-IDF
        text_features = self.text_vectorizer.transform(texts).toarray()

        # Scale and weight stars
        star_features = self.scaler.transform(stars) * self.star_weight

        # Combine features
        combined_features = np.hstack([text_features, star_features])

        return combined_features

In [ ]:
class IMDBSentimentAnalyzer:
    def __init__(self, star_weight=0.1):
        self.preprocessor = SpacyTextPreprocessor()
        self.feature_combiner = FeatureCombiner(star_weight=star_weight)
        self.model = LogisticRegression(
            random_state=42,
            max_iter=1000,
            C=1.0,
            class_weight='balanced'
        )
        self.star_weight = star_weight
        self.fitted = False

    # -----------------------------------------------------------
    # DATA PREPARATION
    # -----------------------------------------------------------
    def prepare_data(self, reviews, stars):
        """Preprocess reviews and pair them with star ratings."""
        cleaned_reviews = self.preprocessor.transform(reviews)
        prepared_data = list(zip(cleaned_reviews, stars))
        return prepared_data

    # -----------------------------------------------------------
    # TRAINING
    # -----------------------------------------------------------
    def fit(self, data, labels):
        """Fit the model with prepared data and labels."""
        print(f"Training with star weight: {self.star_weight}")
        print(f"Processing {len(data)} reviews...")

        # Fit and transform features
        print("Fitting feature combiner...")
        self.feature_combiner.fit(data)

        print("Transforming features...")
        X_features = self.feature_combiner.transform(data)

        print(f"Feature matrix shape: {X_features.shape}")
        print(f"Text features: {X_features.shape[1] - 1}")
        print(f"Star features: 1 (weighted by {self.star_weight})")

        print("Training logistic regression model...")
        self.model.fit(X_features, labels)
        self.fitted = True

        print("Model training complete.")
        return self

    # -----------------------------------------------------------
    # PREDICTION
    # -----------------------------------------------------------
    def _prepare_features(self, reviews, stars):
        """Helper function to preprocess and vectorize new data."""
        data = self.prepare_data(reviews, stars)
        X_features = self.feature_combiner.transform(data)
        return X_features

    def predict(self, reviews, stars):
        if not self.fitted:
            raise ValueError("Model must be fitted before prediction")

        X_features = self._prepare_features(reviews, stars)
        return self.model.predict(X_features)

    def predict_proba(self, reviews, stars):
        if not self.fitted:
            raise ValueError("Model must be fitted before prediction")

        X_features = self._prepare_features(reviews, stars)
        return self.model.predict_proba(X_features)

    # -----------------------------------------------------------
    # EVALUATION
    # -----------------------------------------------------------
    def evaluate_model(self, test_df, true_labels):
        """Evaluate the model using a test DataFrame with ['review', 'stars']."""
        if not self.fitted:
            raise ValueError("Model must be fitted before evaluation")

        reviews = test_df["review"].values
        stars = test_df["stars"].values

        print("Evaluating model on test data...")
        predictions = self.predict(reviews, stars)
        probabilities = self.predict_proba(reviews, stars)

        accuracy = accuracy_score(true_labels, predictions)
        f1 = f1_score(true_labels, predictions)

        print(f"Accuracy: {accuracy:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print("\nClassification Report:")
        print(classification_report(true_labels, predictions, target_names=['Negative', 'Positive']))

        return {
            'accuracy': accuracy,
            'f1_score': f1,
            'predictions': predictions,
            'probabilities': probabilities
        }

    # -----------------------------------------------------------
    # FEATURE IMPORTANCE
    # -----------------------------------------------------------
    def get_feature_importance(self):
        if not self.fitted:
            raise ValueError("Model must be fitted before retrieving feature importance")

        coefs = self.model.coef_[0]
        text_features = self.feature_combiner.text_vectorizer.get_feature_names_out()
        all_features = list(text_features) + ['star_rating']

        importance_df = pd.DataFrame({
            'feature': all_features,
            'importance': np.abs(coefs)
        }).sort_values('importance', ascending=False)

        return importance_df

    # -----------------------------------------------------------
    # SAVE / LOAD
    # -----------------------------------------------------------
    def save_model(self, filepath):
        if not self.fitted:
            raise ValueError("Model must be fitted before saving")

        model_data = {
            'preprocessor': self.preprocessor,
            'feature_combiner': self.feature_combiner,
            'model': self.model,
            'star_weight': self.star_weight,
            'fitted': self.fitted
        }

        with open(filepath, 'wb') as f:
            pickle.dump(model_data, f)

        print(f"Model saved to {filepath}")

    def load_model(self, filepath):
        with open(filepath, 'rb') as f:
            model_data = pickle.load(f)

        self.preprocessor = model_data['preprocessor']
        self.feature_combiner = model_data['feature_combiner']
        self.model = model_data['model']
        self.star_weight = model_data['star_weight']
        self.fitted = model_data['fitted']

        print(f"Model loaded from {filepath}")

### Function to Load The data

In [ ]:
def load_imdb_data(filepath):
    try:
        df = pd.read_csv(filepath)

        # Check required columns
        if 'review' not in df.columns:
            raise ValueError("Dataset must have 'review' column")
        if 'sentiment' not in df.columns:
            raise ValueError("Dataset must have 'sentiment' column")

        df["sentiment"] = df["sentiment"].map({"negative": 0, "positive": 1})
        df_positive = df[df['sentiment']==1]
        df_negative = df[df['sentiment']==0]
        df_positive = df_positive.sort_values('stars', ascending=False)
        df_negative = df_negative.sort_values('stars', ascending=True)
        df = pd.concat([df_positive, df_negative])

        # Add synthetic star ratings if not present
        if 'stars' not in df.columns:
            print("Adding synthetic star ratings based on sentiment...")
            # Positive reviews: 6-10 stars, Negative reviews: 1-5 stars
            df['stars'] = df['sentiment'].apply(
                lambda x: np.random.randint(7, 11) if x == 1 else np.random.randint(1, 5)
            )

        print(f"Loaded dataset with {len(df)} reviews")
        print(f"Sentiment distribution:")
        print(df['sentiment'].value_counts())
        print(f"Star rating distribution:")
        print(df['stars'].value_counts().sort_index())

        return df

    except FileNotFoundError:
        print(f"File {filepath} not found.")


### Loading data

In [ ]:
df = load_imdb_data("data/review.csv")

### Splitting data

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['sentiment']
)

print(f"Training set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

### Setting weight and initialising

In [ ]:
weight = 0.0048
analyzer = IMDBSentimentAnalyzer(star_weight=weight)

### Preparing data

In [ ]:
prepared_data = analyzer.prepare_data(train_df['review'].values, train_df['stars'].values)

In [ ]:
np.array_equal(prepared_data, train_df['sentiment'].values)

In [ ]:
analyzer.X = prepared_data
labels = train_df['sentiment'].values

### Fitting the model

In [ ]:
analyzer.fit(
    prepared_data,
    labels
)

### Evaluating

In [ ]:
results = analyzer.evaluate_model(test_df[["review", "stars"]], test_df["sentiment"])

In [ ]:
results

### Helper function for single prediction

In [ ]:
def predict_sentiment(analyzer, review_text, star_rating):
    prob = analyzer.predict_proba([review_text], [star_rating])[0]
    prediction = analyzer.predict([review_text], [star_rating])[0]

    return {
        'prediction': 'Positive' if prediction == 1 else 'Negative',
        'confidence': {
            'negative': prob[0],
            'positive': prob[1]
        },
        'review': review_text,
        'stars': star_rating
    }

In [ ]:
def print_review(result):
    print(f"Review: \"{result['review']}\"")
    print(f"Stars: {result['stars']}/10")
    print(f"Primary Prediction: {result['prediction']}")
    print(f"Probabilities:")
    print(f"  Negative: {result['confidence']['negative']*100}%")
    print(f"  Positive: {result['confidence']['positive']*100}%")

In [ ]:
print("\nExample Usage:")
sample_review = "This movie was incredible with amazing special effects!"
sample_stars = 10

result = predict_sentiment(analyzer, sample_review, sample_stars)
print_review(result)

In [ ]:
print("\nTop 20 Most Important Features:")
importance_df = analyzer.get_feature_importance()
importance_df.head(20)

### Saving model

In [ ]:
analyzer.save_model("Sentiment_model.pkl")